In [ ]:
# Jupyter Notebook content for EDA - save as a .ipynb file

"""
MediPredict AI - Exploratory Data Analysis
Healthcare Readmission Prediction
"""

# %% [markdown]
# # MediPredict AI: Exploratory Data Analysis
# ## Hospital Readmission Prediction

# %% [code]
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# %% [code]
# Load data
df = pd.read_csv('../data/healthcare_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# %% [code]
# Basic statistics
print("Basic Statistics:")
display(df.describe())

# %% [code]
# Target variable analysis
print("\nTarget Distribution (Readmission within 30 days):")
readmission_rate = df['readmitted_30d'].mean()
print(f"Readmission Rate: {readmission_rate:.1%}")
print(df['readmitted_30d'].value_counts(normalize=True))

# %% [code]
# Age distribution by readmission
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df[df['readmitted_30d']==0]['age'].hist(alpha=0.5, label='Not Readmitted', bins=30, ax=axes[0])
df[df['readmitted_30d']==1]['age'].hist(alpha=0.5, label='Readmitted', bins=30, ax=axes[0])
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution by Readmission Status')
axes[0].legend()

# Box plot
df.boxplot(column='age', by='readmitted_30d', ax=axes[1])
axes[1].set_title('Age Box Plot by Readmission Status')
axes[1].set_xlabel('Readmitted')
axes[1].set_ylabel('Age')

plt.tight_layout()
plt.show()

# %% [code]
# Correlation heatmap of numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.show()

# %% [code]
# Top correlations with readmission
corr_with_target = correlation_matrix['readmitted_30d'].sort_values(ascending=False)
print("Top 10 features correlated with readmission:")
display(corr_with_target.head(10))

# %% [code]
# Medical conditions impact
conditions = ['diabetes', 'hypertension', 'heart_disease', 'copd', 'kidney_disease', 'cancer']
condition_readmission = pd.DataFrame()

for condition in conditions:
    readmit_rate = df.groupby(condition)['readmitted_30d'].mean()
    condition_readmission[condition] = [readmit_rate[0], readmit_rate[1]]

condition_readmission.index = ['Without Condition', 'With Condition']

# Plot
condition_readmission.T.plot(kind='bar', figsize=(12, 6))
plt.title('Readmission Rate by Medical Condition', fontsize=14)
plt.xlabel('Medical Condition')
plt.ylabel('Readmission Rate')
plt.legend(title='')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# %% [code]
# Length of stay analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
df[df['readmitted_30d']==0]['length_of_stay'].hist(alpha=0.5, label='Not Readmitted', bins=30, ax=axes[0])
df[df['readmitted_30d']==1]['length_of_stay'].hist(alpha=0.5, label='Readmitted', bins=30, ax=axes[0])
axes[0].set_xlabel('Length of Stay (days)')
axes[0].set_ylabel('Count')
axes[0].set_title('LOS Distribution by Readmission Status')
axes[0].legend()

# Box plot
df.boxplot(column='length_of_stay', by='readmitted_30d', ax=axes[1])
axes[1].set_title('LOS Box Plot by Readmission Status')
axes[1].set_xlabel('Readmitted')
axes[1].set_ylabel('Length of Stay (days)')

plt.tight_layout()
plt.show()

# %% [code]
# Lab score analysis
plt.figure(figsize=(10, 6))
df.boxplot(column='lab_score', by='readmitted_30d')
plt.title('Lab Score by Readmission Status')
plt.xlabel('Readmitted')
plt.ylabel('Lab Score')
plt.show()

# %% [code]
# Prior hospital visits impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prior hospital visits
df.groupby('prior_hospital_visits_1yr')['readmitted_30d'].mean().plot(kind='bar', ax=axes[0])
axes[0].set_title('Readmission Rate by Prior Hospital Visits')
axes[0].set_xlabel('Prior Hospital Visits (1 year)')
axes[0].set_ylabel('Readmission Rate')

# Prior ED visits
df.groupby('prior_emergency_visits_1yr')['readmitted_30d'].mean().plot(kind='bar', ax=axes[1])
axes[1].set_title('Readmission Rate by Prior ED Visits')
axes[1].set_xlabel('Prior ED Visits (1 year)')
axes[1].set_ylabel('Readmission Rate')

plt.tight_layout()
plt.show()

# %% [code]
# Comorbidity analysis
df['comorbidity_count'] = df[conditions].sum(axis=1)
comorbidity_readmission = df.groupby('comorbidity_count')['readmitted_30d'].mean()

plt.figure(figsize=(10, 6))
comorbidity_readmission.plot(kind='bar')
plt.title('Readmission Rate by Number of Comorbidities', fontsize=14)
plt.xlabel('Number of Comorbidities')
plt.ylabel('Readmission Rate')
plt.xticks(range(0, 7))
plt.grid(axis='y', alpha=0.3)
plt.show()

# %% [code]
# Interactive dashboard preview with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Age Distribution', 'Readmission by Age Group', 
                    'Length of Stay Impact', 'Comorbidity Impact'),
    specs=[[{'type': 'histogram'}, {'type': 'bar'}],
           [{'type': 'box'}, {'type': 'scatter'}]]
)

# Age histogram
fig.add_trace(go.Histogram(x=df['age'], nbinsx=30, name='Age Distribution'), row=1, col=1)

# Age group readmission
df['age_group'] = pd.cut(df['age'], bins=[0, 40, 65, 80, 100], labels=['<40', '40-65', '65-80', '80+'])
age_readmit = df.groupby('age_group')['readmitted_30d'].mean().reset_index()
fig.add_trace(go.Bar(x=age_readmit['age_group'], y=age_readmit['readmitted_30d'], name='Readmission Rate'), row=1, col=2)

# LOS box plot
fig.add_trace(go.Box(x=df['readmitted_30d'], y=df['length_of_stay'], name='LOS by Readmission'), row=2, col=1)

# Comorbidity scatter
comorbidity_avg = df.groupby('comorbidity_count')['readmitted_30d'].mean().reset_index()
fig.add_trace(go.Scatter(x=comorbidity_avg['comorbidity_count'], y=comorbidity_avg['readmitted_30d'], 
                         mode='lines+markers', name='Comorbidity Impact'), row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text="MediPredict AI - EDA Dashboard")
fig.show()

# %% [code]
# Summary statistics for competition report
print("\n" + "="*50)
print("EDA SUMMARY FOR COMPETITION REPORT")
print("="*50)

print(f"\n1. Dataset Overview:")
print(f"   - Total patients: {len(df):,}")
print(f"   - Features: {len(df.columns)}")
print(f"   - Readmission rate: {readmission_rate:.1%}")

print(f"\n2. Key Risk Factors Identified:")
top_factors = corr_with_target.head(5).index.tolist()
top_values = corr_with_target.head(5).values.tolist()
for factor, value in zip(top_factors, top_values):
    print(f"   - {factor}: {value:.3f}")

print(f"\n3. High-Risk Patient Profile:")
high_risk = df[df['readmitted_30d'] == 1]
print(f"   - Average age: {high_risk['age'].mean():.1f} years")
print(f"   - Average LOS: {high_risk['length_of_stay'].mean():.1f} days")
print(f"   - Average comorbidities: {high_risk['comorbidity_count'].mean():.1f}")
print(f"   - Prior hospital visits: {high_risk['prior_hospital_visits_1yr'].mean():.1f}")

print(f"\n4. Model Implications:")
print(f"   - Recall is critical (identifying high-risk patients)")
print(f"   - SHAP explanations will focus on top 5-10 features")
print(f"   - Dashboard should highlight modifiable risk factors")